In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, datediff, lit
from os.path import basename, abspath

In [2]:
problem_name = basename(abspath("."))

In [3]:
spark = SparkSession.builder.appName(problem_name).getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/06 00:33:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
employees = spark.read\
    .option("header", "true")\
    .option("inferSchema", "true")\
    .format("csv")\
    .load("files/employees.txt")

In [5]:
employees.show()

+------+-----+----------+
|emp_id| name| join_date|
+------+-----+----------+
|     1| John|2015-03-12|
|     2|Alice|2018-05-20|
|     3|  Bob|2010-06-14|
|     4|Carol|2019-07-08|
|     5| Dave|2013-02-25|
+------+-----+----------+



In [6]:
employees.createOrReplaceTempView("employees")

In [7]:
result = employees\
    .withColumn("date_diff", datediff(lit("2020-05-01").cast("date"), col("join_date"))/365)\
    .filter(col("date_diff")>5)\
    .select(employees.columns)

In [8]:
result.show()

+------+----+----------+
|emp_id|name| join_date|
+------+----+----------+
|     1|John|2015-03-12|
|     3| Bob|2010-06-14|
|     5|Dave|2013-02-25|
+------+----+----------+



In [9]:
spark.sql(
    """
    select *
    from employees
    where datediff(cast('2020-05-01' as date), join_date)/365 > 5
    """
).show()

+------+----+----------+
|emp_id|name| join_date|
+------+----+----------+
|     1|John|2015-03-12|
|     3| Bob|2010-06-14|
|     5|Dave|2013-02-25|
+------+----+----------+



In [10]:
spark.stop()